# Trial Analysis — Eagle Overnight Run (2026-04-11)

Optimization convergence, fitness distributions, and pruning behavior for the first eagle optimization run.

**Run details**: 724 trials over ~14 hours, single instance (pre-async-dispatch), TPE sampler with median pruner, 10 active opponents.

In [ ]:
import sys
sys.path.insert(0, "../src")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from pathlib import Path
from optuna.trial import TrialState

EXPERIMENT_DIR = Path("../experiments/eagle-overnight-2026-04-11")
STUDY_DB = EXPERIMENT_DIR / "study.db"
EVAL_LOG = EXPERIMENT_DIR / "evaluation_log.jsonl"

study = optuna.load_study(study_name="eagle", storage=f"sqlite:///{STUDY_DB}")
trials = study.trials

# Load eval log
eval_entries = []
with open(EVAL_LOG) as f:
    for line in f:
        eval_entries.append(json.loads(line))

print(f"Study: {len(trials)} trials")
print(f"Eval log: {len(eval_entries)} entries")
print(f"States: {dict(pd.Series([t.state.name for t in trials]).value_counts())}")

## Fitness Distribution

In [ ]:
completed = [t for t in trials if t.state == TrialState.COMPLETE]
pruned = [t for t in trials if t.state == TrialState.PRUNED]
fitness_values = [t.value for t in completed]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(fitness_values, bins=30, edgecolor="black", alpha=0.7)
axes[0].axvline(np.mean(fitness_values), color="red", linestyle="--", label=f"Mean: {np.mean(fitness_values):.3f}")
axes[0].axvline(np.median(fitness_values), color="orange", linestyle="--", label=f"Median: {np.median(fitness_values):.3f}")
axes[0].set_xlabel("Fitness")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Fitness Distribution ({len(completed)} completed trials)")
axes[0].legend()

# Box plot with quartiles
bp = axes[1].boxplot(fitness_values, vert=True, widths=0.6)
axes[1].set_ylabel("Fitness")
axes[1].set_title("Fitness Quartiles")
q1, q3 = np.percentile(fitness_values, [25, 75])
axes[1].text(1.15, q1, f"Q1: {q1:.3f}", va="center")
axes[1].text(1.15, q3, f"Q3: {q3:.3f}", va="center")
axes[1].text(1.15, np.median(fitness_values), f"Med: {np.median(fitness_values):.3f}", va="center")

plt.tight_layout()
plt.show()

# Summary stats
print(f"Completed: {len(completed)}, Pruned: {len(pruned)}")
print(f"Min: {min(fitness_values):.4f}, Max: {max(fitness_values):.4f}")
print(f"Mean: {np.mean(fitness_values):.4f}, Std: {np.std(fitness_values):.4f}")
print(f"Fitness = 1.0 count: {sum(1 for v in fitness_values if v >= 0.999)}")
print(f"Fitness > 0.5 count: {sum(1 for v in fitness_values if v > 0.5)}")
print(f"Fitness < 0.1 count: {sum(1 for v in fitness_values if v < 0.1)}")

## Optimization Convergence

Best fitness over trial number and wall-clock time.

In [ ]:
# Build convergence curves from completed trials ordered by number
completed_sorted = sorted(completed, key=lambda t: t.number)
trial_nums = [t.number for t in completed_sorted]
values = [t.value for t in completed_sorted]

# Running best
running_best = []
best_so_far = float("-inf")
for v in values:
    best_so_far = max(best_so_far, v)
    running_best.append(best_so_far)

# Running mean (window=20)
window = 20
running_mean = pd.Series(values).rolling(window, min_periods=1).mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Individual trials + running best
axes[0].scatter(trial_nums, values, alpha=0.3, s=10, label="Individual trials")
axes[0].plot(trial_nums, running_best, color="red", linewidth=2, label="Best so far")
axes[0].set_ylabel("Fitness")
axes[0].set_title("Optimization Convergence")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Running mean
axes[1].plot(trial_nums, running_mean, color="blue", linewidth=1.5, label=f"Rolling mean (window={window})")
axes[1].fill_between(trial_nums,
                      pd.Series(values).rolling(window, min_periods=1).quantile(0.25),
                      pd.Series(values).rolling(window, min_periods=1).quantile(0.75),
                      alpha=0.2, label="IQR")
axes[1].set_xlabel("Trial Number")
axes[1].set_ylabel("Fitness")
axes[1].set_title("Rolling Fitness Statistics")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# When did the best fitness first appear?
best_trial = max(completed, key=lambda t: t.value)
print(f"Best trial: #{best_trial.number}, fitness={best_trial.value:.4f}")
first_good = next(i for i, v in enumerate(values) if v >= 0.999)
print(f"First fitness >= 0.999 at completed trial index {first_good} (trial #{trial_nums[first_good]})")

## Pruning Analysis

How effective is the median pruner? At which rungs are trials pruned?

In [ ]:
# Pruning rung distribution from eval log
pruned_entries = [e for e in eval_entries if e["pruned"]]
completed_entries = [e for e in eval_entries if not e["pruned"]]

pruned_rungs = [e["opponents_evaluated"] for e in pruned_entries]
completed_rungs = [e["opponents_evaluated"] for e in completed_entries]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pruning rung histogram
if pruned_rungs:
    axes[0].hist(pruned_rungs, bins=range(1, 12), edgecolor="black", alpha=0.7, align="left")
    axes[0].set_xlabel("Rung (opponents evaluated)")
    axes[0].set_ylabel("Count")
    axes[0].set_title(f"Pruning Rung Distribution ({len(pruned_entries)} pruned)")
    axes[0].set_xticks(range(1, 11))
else:
    axes[0].text(0.5, 0.5, "No pruned trials in eval log", ha="center", va="center",
                 transform=axes[0].transAxes)

# Trial outcome pie chart
total_trials = len(trials)
states = pd.Series([t.state.name for t in trials]).value_counts()
axes[1].pie(states.values, labels=states.index, autopct="%1.0f%%", startangle=90)
axes[1].set_title(f"Trial Outcomes (n={total_trials})")

plt.tight_layout()
plt.show()

# Pruning efficiency
print(f"Pruned in eval log: {len(pruned_entries)}/{len(eval_entries)} ({100*len(pruned_entries)/len(eval_entries):.0f}%)")
if pruned_rungs:
    print(f"Mean pruning rung: {np.mean(pruned_rungs):.1f}")
    print(f"Median pruning rung: {np.median(pruned_rungs):.1f}")
    # Opponents saved by pruning (each pruned trial saved 10-rung opponents)
    saved = sum(10 - r for r in pruned_rungs)
    print(f"Opponent matchups saved by pruning: {saved}")

## Intermediate Values (Rung Progression)

How does cumulative fitness evolve across rungs for completed vs pruned trials?

In [ ]:
# Plot intermediate values from Optuna study
fig, ax = plt.subplots(figsize=(14, 6))

# Completed trials — plot rung progression
for t in completed[:50]:  # limit to 50 for readability
    steps = sorted(t.intermediate_values.keys())
    vals = [t.intermediate_values[s] for s in steps]
    ax.plot(steps, vals, alpha=0.15, color="blue", linewidth=0.8)

# Pruned trials
for t in pruned[:50]:
    steps = sorted(t.intermediate_values.keys())
    vals = [t.intermediate_values[s] for s in steps]
    ax.plot(steps, vals, alpha=0.15, color="red", linewidth=0.8)

# Best trial
best = max(completed, key=lambda t: t.value)
steps = sorted(best.intermediate_values.keys())
vals = [best.intermediate_values[s] for s in steps]
ax.plot(steps, vals, color="green", linewidth=2.5, label=f"Best (#{best.number}, f={best.value:.3f})")

ax.set_xlabel("Rung (0-indexed)")
ax.set_ylabel("Cumulative Fitness")
ax.set_title("Rung Progression: Completed (blue) vs Pruned (red)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Throughput Over Time

Trial completion rate and wall-clock timing.

In [ ]:
# Parse timestamps from eval log
from datetime import datetime

timestamps = []
for e in eval_entries:
    ts = datetime.fromisoformat(e["timestamp"])
    timestamps.append(ts)

if timestamps:
    start_time = min(timestamps)
    hours_elapsed = [(t - start_time).total_seconds() / 3600 for t in timestamps]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Cumulative trials over time
    axes[0].plot(sorted(hours_elapsed), range(1, len(hours_elapsed) + 1))
    axes[0].set_xlabel("Hours Elapsed")
    axes[0].set_ylabel("Cumulative Trials Evaluated")
    axes[0].set_title("Evaluation Throughput")
    axes[0].grid(True, alpha=0.3)

    # Per-hour throughput
    hour_bins = [int(h) for h in hours_elapsed]
    hourly_counts = pd.Series(hour_bins).value_counts().sort_index()
    axes[1].bar(hourly_counts.index, hourly_counts.values, edgecolor="black", alpha=0.7)
    axes[1].set_xlabel("Hour")
    axes[1].set_ylabel("Trials Evaluated")
    axes[1].set_title("Trials per Hour")
    axes[1].grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.show()

    total_hours = max(hours_elapsed)
    print(f"Total runtime: {total_hours:.1f} hours")
    print(f"Overall throughput: {len(eval_entries) / total_hours:.1f} evals/hour")
else:
    print("No timestamps available")

## Combat Duration Distribution

How long do individual matchups take?

In [ ]:
# Extract all individual matchup durations
durations = []
outcomes = []
for e in eval_entries:
    for r in e["opponent_results"]:
        durations.append(r["duration_seconds"])
        outcomes.append(r["winner"])

durations = np.array(durations)
outcomes = np.array(outcomes)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Duration histogram
axes[0].hist(durations, bins=40, edgecolor="black", alpha=0.7)
axes[0].axvline(np.mean(durations), color="red", linestyle="--", label=f"Mean: {np.mean(durations):.1f}s")
axes[0].axvline(np.median(durations), color="orange", linestyle="--", label=f"Median: {np.median(durations):.1f}s")
axes[0].set_xlabel("Duration (seconds)")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Combat Duration ({len(durations)} matchups)")
axes[0].legend()

# Duration by outcome
for outcome in ["PLAYER", "ENEMY", "STOPPED"]:
    mask = outcomes == outcome
    if mask.any():
        axes[1].hist(durations[mask], bins=30, alpha=0.5, label=f"{outcome} ({mask.sum()})")
axes[1].set_xlabel("Duration (seconds)")
axes[1].set_ylabel("Count")
axes[1].set_title("Duration by Outcome")
axes[1].legend()

plt.tight_layout()
plt.show()

# Outcome summary
outcome_counts = pd.Series(outcomes).value_counts()
print("Outcome distribution:")
for outcome, count in outcome_counts.items():
    pct = 100 * count / len(outcomes)
    mean_dur = durations[outcomes == outcome].mean()
    print(f"  {outcome}: {count} ({pct:.0f}%), mean duration={mean_dur:.1f}s")